# 📊 Auditoría y Análisis Exploratorio de Datos (EDA) — SEC Datasets (2010–2023)

Este notebook realiza una **auditoría completa de integridad y consistencia temporal** sobre los datasets regulatorios descargados desde Cloudflare R2:
- **10-K / 10-Q**: Informes periódicos anuales y trimestrales obligatorios (336 filings).
- **8-K**: Hechos relevantes y noticias extraordinarias con texto narrativo para FinBERT (5,392 eventos).

### Objetivos de la Auditoría:
1. **Verificación de Cobertura Temporal**: Comprobar que no existen lagunas temporales ni años vacíos (2010 a 2023).
2. **Simetría entre Bancos**: Verificar que los 6 bancos del universo Big 6 (`JPM`, `BAC`, `WFC`, `C`, `GS`, `MS`) tienen exactamente sus 4 reportes periódicos anuales.
3. **Análisis de Eventos 8-K**: Distribución de eventos por categoría regulatoria (`Item 2.02`, `Item 8.01`, etc.) y evolución histórica de volumen.
4. **Ventanas Horarias de Publicación (`acceptance_date_time`)**: Distribución horaria para alinear el cruce de sentimiento con la sesión bursátil (Pre-Market, Regular Trading, After-Hours).
5. **Métricas NLP**: Análisis de longitud de textos y palabras para entrada en modelos Transformer.

In [ ]:
import sys
from pathlib import Path
import io
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from botocore.config import Config

# Estilo visual profesional para reportes y visualizaciones
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# 1. Resolver ruta hacia 'src'
CURRENT_DIR = Path.cwd().resolve()
SRC_PATH = str(CURRENT_DIR.parent) if CURRENT_DIR.name == "notebooks" else str(CURRENT_DIR / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from config.settings import settings

# 2. Conexión S3 con Cloudflare R2
s3 = boto3.client(
    "s3",
    endpoint_url=settings.s3_endpoint_url,
    aws_access_key_id=settings.aws_access_key_id,
    aws_secret_access_key=settings.aws_secret_access_key,
    region_name="auto",
    config=Config(s3={"addressing_style": "path"})
)

def leer_parquet(key: str) -> pd.DataFrame:
    """Descarga y deserializa directamente el Parquet de R2 en un DataFrame de Pandas."""
    obj = s3.get_object(Bucket=settings.s3_bucket_name, Key=key.lstrip("/"))
    return pd.read_parquet(io.BytesIO(obj["Body"].read()))

# 3. Cargar ambos datasets
df_10k = leer_parquet("raw/sec/10k-10q-2010-2023.parquet")
df_8k = leer_parquet("raw/sec/8k-2010-2023.parquet")

# Parsear fechas
df_10k["filing_date"] = pd.to_datetime(df_10k["filing_date"])
df_10k["year"] = df_10k["filing_date"].dt.year

df_8k["filing_date"] = pd.to_datetime(df_8k["filing_date"])
df_8k["acceptance_dt"] = pd.to_datetime(df_8k["acceptance_date_time"])
df_8k["year"] = df_8k["filing_date"].dt.year
df_8k["char_count"] = df_8k["content"].str.len()
df_8k["word_count"] = df_8k["content"].str.split().str.len()

print(f" Dataset 10-K / 10-Q cargado: {df_10k.shape[0]:,} filas | Columnas: {list(df_10k.columns)}")
print(f" Dataset 8-K cargado: {df_8k.shape[0]:,} eventos | Columnas: {list(df_8k.columns)}")

## 1. Verificación de Integridad: Informes Periódicos (10-K y 10-Q)
Por regulación bursátil en EE.UU., cada entidad bancaria debe presentar exactamente **4 informes obligatorios al año**:
- **3 trimestrales** (Form 10-Q)
- **1 anual auditado** (Form 10-K)

Verificamos la matriz de frecuencias por banco y por año (2010 a 2023):

In [ ]:
# Matriz pivote de reportes periódicos (Año x Banco)
matriz_10k = pd.crosstab(df_10k["year"], df_10k["ticker"])

# Visualización con Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(matriz_10k, annot=True, cmap="Blues", cbar=False, fmt="d", linewidths=1)
plt.title("Matriz de Consistencia 10-K / 10-Q por Año y Banco (Debe ser exactamente 4 al año)", fontsize=13, fontweight="bold")
plt.xlabel("Entidad Financiera (Ticker)")
plt.ylabel("Año de Publicación")
plt.show()

# Comprobación matemática de integridad total
es_perfecto = (matriz_10k == 4).all().all()
print(f"¿Integridad 10-K/10-Q perfecta sin lagunas (14 años x 6 bancos x 4 reportes = 336)? {' SÍ (100% COMPLETO)' if es_perfecto else '❌ FALTAN DATOS'}")

## 2. Cobertura Histórica de Hechos Relevantes (8-K)
Analizamos la distribución de eventos no programados a lo largo del periodo 2010–2023. Esto nos permite comprobar que **ningún año quedó sin cobertura** y cómo varió la actividad regulatoria por entidad.

In [ ]:
# 1. Gráfico de evolución temporal por banco
matriz_8k = pd.crosstab(df_8k["year"], df_8k["ticker"])

fig, ax = plt.subplots(figsize=(14, 6))
matriz_8k.plot(kind="bar", stacked=True, ax=ax, colormap="tab10", edgecolor="black", alpha=0.85)
plt.title("Volumen Anual de Hechos Relevantes 8-K por Entidad Financiera (2010–2023)", fontsize=14, fontweight="bold")
plt.xlabel("Año")
plt.ylabel("Número de Eventos 8-K")
plt.legend(title="Banco", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 2. Tabla resumen por banco
resumen_banco = df_8k.groupby("ticker").agg(
    total_eventos=("item", "count"),
    filings_unicas=("accession_number", "nunique"),
    primer_evento=("filing_date", "min"),
    ultimo_evento=("filing_date", "max"),
)
resumen_banco["eventos_por_filing"] = (resumen_banco["total_eventos"] / resumen_banco["filings_unicas"]).round(2)
display(resumen_banco.sort_values("total_eventos", ascending=False))

## 3. Análisis de Tipología de Eventos SEC (Taxonomía de Ítems)
Los reportes 8-K se dividen en códigos regulatorios estandarizados:
- **Item 2.02**: Resultados operativos trimestrales (*Earnings releases*) — Máximo impacto en volatilidad.
- **Item 8.01**: Otros hechos materiales relevantes del negocio.
- **Item 5.02**: Salida, nombramiento o renuncia de directivos / consejeros clave (CEO, CFO).
- **Item 7.01**: Divulgación de información según *Regulation Fair Disclosure* (FD).
- **Item 9.01**: Anexos y notas de prensa oficiales adjuntas.

In [ ]:
# Conteo de ítems más frecuentes
top_items = df_8k["item"].value_counts().head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_items.values, y=top_items.index, hue=top_items.index, legend=False, palette="viridis")
plt.title("Top 10 Categorías de Hechos Relevantes 8-K en el Universo Bancario", fontsize=13, fontweight="bold")
plt.xlabel("Total Eventos Registrados")
plt.ylabel("Código de Ítem SEC")
for index, value in enumerate(top_items.values):
    plt.text(value + 20, index, f"{value:,} ({value/len(df_8k)*100:.1f}%)", va="center", fontweight="bold")
plt.show()

## 4. Distribución Horaria de Publicación (`acceptance_date_time`)
**Dato crítico para el modelo predictivo de bolsa:**
¿A qué hora publica la SEC los reportes? Para cruzar el sentimiento con las cotizaciones de Wall Street sin *data leakage*:
- La sesión bursátil oficial abre a las **09:30 ET** (14:30 UTC / 13:30 en verano) y cierra a las **16:00 ET** (21:00 UTC / 20:00 en verano).
- Las publicaciones fuera de mercado deben asociarse con la apertura del día siguiente.

In [ ]:
# Extraer la hora UTC de aceptación en la SEC
df_8k["hora_utc"] = df_8k["acceptance_dt"].dt.hour

plt.figure(figsize=(14, 5))
sns.countplot(data=df_8k, x="hora_utc", hue="hora_utc", legend=False, palette="crest", edgecolor="black")
plt.title("Distribución de Publicaciones SEC 8-K por Hora del Día (UTC)", fontsize=14, fontweight="bold")
plt.xlabel("Hora de Aceptación Oficial SEC (UTC)")
plt.ylabel("Número de Publicaciones")

# Añadir anotaciones de mercado
plt.axvline(x=14.5, color="red", linestyle="--", linewidth=2, label="Apertura NYSE (09:30 ET ~ 14:30 UTC)")
plt.axvline(x=21.0, color="orange", linestyle="--", linewidth=2, label="Cierre NYSE (16:00 ET ~ 21:00 UTC)")
plt.legend(loc="upper right")
plt.show()

# Resumen de sesión de publicación
def clasificar_sesion(h):
    if 14 <= h < 21:
        return "Durante Sesión Bursátil (Mercado Abierto)"
    elif 10 <= h < 14:
        return "Pre-Market (Antes de Apertura)"
    else:
        return "After-Hours / Noche (Tras Cierre Bursátil)"

df_8k["sesion_mercado"] = df_8k["hora_utc"].apply(clasificar_sesion)
display(df_8k["sesion_mercado"].value_counts().to_frame("Total Eventos"))

## 5. Auditoría de Texto y Métricas NLP (Preparación para FinBERT)
FinBERT tiene un límite de entrada de tokens (512 tokens por fragmento). Analizamos la longitud de caracteres y palabras de los hechos relevantes limpios para validar que el texto está listo para inferencia sin truncamiento agresivo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Distribución de palabras (acotado a percentil 95 para visualización legible)
p95 = np.percentile(df_8k["word_count"], 95)
sns.histplot(df_8k[df_8k["word_count"] <= p95]["word_count"], bins=50, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title(f"Distribución de Palabras por Evento 8-K (Hasta Percentil 95 = {p95:.0f} palabras)", fontweight="bold")
axes[0].set_xlabel("Número de Palabras")
axes[0].set_ylabel("Frecuencia")

# 2. Palabras por Banco (Boxplot)
sns.boxplot(data=df_8k[df_8k["word_count"] <= p95], x="ticker", y="word_count", ax=axes[1], hue="ticker", legend=False, palette="Set2")
axes[1].set_title("Longitud de Textos de Hechos Relevantes por Banco", fontweight="bold")
axes[1].set_xlabel("Ticker")
axes[1].set_ylabel("Número de Palabras")

plt.tight_layout()
plt.show()

print("Estadísticas de longitud de texto (palabras):")
display(df_8k["word_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame("Métricas"))

## 6. Muestra de Validación: Cruce de Eventos Críticos (Earnings / Item 2.02)
Aislamos una muestra de eventos `Item 2.02` (los de mayor sensibilidad para la predicción de precios) para visualizar su estructura definitiva lista para el pipeline de Machine Learning:

In [ ]:
# Filtrar solo eventos de resultados trimestrales (Item 2.02)
earnings_sample = df_8k[df_8k["item"] == "Item 2.02"][
    ["ticker", "company_name", "filing_date", "acceptance_date_time", "item", "item_description", "content", "document_url"]
].sort_values("filing_date", ascending=False).head(5)

display(earnings_sample)

# Ver el texto real limpio del primer evento de earnings
print("\n--- TEXTO EXTRACTO REAL DEL PRIMER EVENTO (ITEM 2.02) ---")
print(f"Banco: {earnings_sample.iloc[0]['ticker']} | Fecha: {earnings_sample.iloc[0]['filing_date']}")
print(f"Hora aceptación SEC: {earnings_sample.iloc[0]['acceptance_date_time']}")
print(f"Texto: {earnings_sample.iloc[0]['content'][:400]}...")